# RAG Application — No Pinecone Version

**RAG = Retrieval Augmented Generation**


Flow:
1. Search Wikipedia for a topic
2. Split text into chunks
3. Embed with HuggingFace (local)
4. Store in FAISS (in-memory)
5. User asks a question → retrieve relevant chunks → Groq LLM answers


In [ ]:
# ============================================
# INSTALL REQUIRED LIBRARIES
# ============================================
'''
!pip uninstall -y langchain langchain-core langchain-community \\
langchain-text-splitters langchain-groq pydantic faiss-cpu

!pip install -q \\
langchain==0.2.10 \\
langchain-core==0.2.22 \\
langchain-community==0.2.9 \\
langchain-text-splitters==0.2.2 \\
langchain-groq==0.1.5 \\
pydantic==2.8.2 \\
faiss-cpu \\
sentence-transformers \\
transformers \\
huggingface_hub \\
wikipedia

# Restart runtime after install
import os
os.kill(os.getpid(), 9)

'''

'\n!pip uninstall -y langchain langchain-core langchain-community \\\nlangchain-text-splitters langchain-groq pydantic faiss-cpu\n\n!pip install -q \\\nlangchain==0.2.10 \\\nlangchain-core==0.2.22 \\\nlangchain-community==0.2.9 \\\nlangchain-text-splitters==0.2.2 \\\nlangchain-groq==0.1.5 \\\npydantic==2.8.2 \\\nfaiss-cpu \\\nsentence-transformers \\\ntransformers \\\nhuggingface_hub \\\nwikipedia\n\n# Restart runtime after install\nimport os\nos.kill(os.getpid(), 9)\n\n'

In [ ]:
import langchain
print(langchain.__version__)

0.2.10


In [ ]:
from langchain_community.document_loaders import WikipediaLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_groq import ChatGroq
from langchain.chains import RetrievalQA
import os

In [ ]:
# Only Groq API key needed now — no Pinecone!
os.environ["GROQ_API_KEY"] = 'groq'

In [ ]:
# Step 1: Load Wikipedia articles
query_topic = input("Enter topic to search on Wikipedia: ")

loader = WikipediaLoader(
    query=query_topic,
    load_max_docs=3
)

documents = loader.load()
print(f"Loaded {len(documents)} Wikipedia article(s)")
print(documents[0].page_content[:500])

Enter topic to search on Wikipedia: paul


JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [ ]:
# Step 2: Split into chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = text_splitter.split_documents(documents)
print(f"Total chunks: {len(chunks)}")

Total chunks: 28


In [ ]:
# Step 3: Create embeddings (local, no API needed)
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)
print("Embeddings model loaded")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embeddings model loaded


In [ ]:
# Step 4: Build FAISS vector store in memory (replaces Pinecone)
vectorstore = FAISS.from_documents(chunks, embeddings)
print(f"FAISS index built with {vectorstore.index.ntotal} vectors")

FAISS index built with 28 vectors


In [ ]:
# Step 5: Create retriever
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 3}
)

In [ ]:
# Step 6: Load Groq LLM
llm = ChatGroq(
    model_name="llama-3.1-8b-instant",
    temperature=0
)

In [ ]:
# Step 7: Build QA chain
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever,
    chain_type="stuff"
)
print("QA chain ready!")

QA chain ready!


In [ ]:
# Step 8: Ask questions!
query = input("Ask a question: ")

response = qa_chain.invoke(query)
print("\nAnswer:\n", response["result"])

In [ ]:
# Step 8: Ask questions!
query = input("Ask a question: ")

response = qa_chain.invoke(query)
print("\nAnswer:\n", response["result"])

Ask a question: teacher

Answer:
 A teacher is an individual who practices the act of teaching, which is the process of transmitting skills, knowledge, and interpersonal skills to a learner, student, or audience of an educational institution.
